Member 1 — Decision Tree Classifier
ML Assignment: FIFA Player Position Prediction

## Dataset Description

- **Name:** FIFA 22 Complete Player Dataset
- **Source:** https://www.kaggle.com/datasets/stefanoleone992/fifa-22-complete-player-dataset
- **Size:** ~19,000 players, 100+ attributes
- **Task:** Predict player position (ATTACKER, MIDFIELDER, DEFENDER)
- **Features used:** age, height_cm, weight_kg, overall, potential, 
                     pace, shooting, passing, dribbling, defending, physic
- **Target:** position_category (3 classes)
- **Context:** FIFA 22 is a football simulation game. Player 
               statistics reflect real-world player abilities.


**Algorithm:** Decision Tree
**Dataset:** FIFA 22 Players
**Task:** Predict if a player is ATTACKER, MIDFIELDER, or DEFENDER

**BEFORE RUNNING THIS:** Make sure you have run `STEP0_Preprocessing_Shared.ipynb` first and have these files in the same folder:
- `X_train.npy`, `X_test.npy`, `y_train.npy`, `y_test.npy`, `class_names.npy`

---

## 1. Introduction to Decision Tree

A **Decision Tree** is a machine learning algorithm that works exactly like a flowchart.

Imagine you are guessing a player's position:
- **Is their defending score > 70?** → Yes → Likely DEFENDER
- **Is their shooting score > 70?** → Yes → Likely ATTACKER
- Otherwise → MIDFIELDER

The algorithm automatically finds the best questions to ask from the data.

**Why Decision Tree for this task?**
- Easy to understand and explain (great for a report!)
- Works well with player statistics
- No need to scale data (though we already did it)
- Can be visualized as an actual tree diagram

## 2. Import libraries

In [ ]:
# Import needed libraries
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Decision Tree from scikit-learn
from sklearn.tree import DecisionTreeClassifier, plot_tree

# Evaluation tools
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

print('Libraries imported!')

Libraries imported!


## 3. Load the preprocessed data

In [ ]:
# Load the shared preprocessed data
X_train = np.load('X_train.npy')
X_test  = np.load('X_test.npy')
y_train = np.load('y_train.npy')
y_test  = np.load('y_test.npy')
class_names = np.load('class_names.npy', allow_pickle=True)

print('Data loaded successfully!')
print(f'Training samples: {len(X_train)}')
print(f'Testing samples:  {len(X_test)}')
print(f'Classes: {list(class_names)}')

Data loaded successfully!
Training samples: 15345
Testing samples:  3422
Classes: ['ATTACKER', 'DEFENDER', 'MIDFIELDER']


## 4. Train the Decision Tree model

In [ ]:
# class_weight='balanced' automatically penalizes mistakes 
# on the minority class (Attacker) more heavily
dt_model = DecisionTreeClassifier(
    max_depth=5, 
    random_state=42,
    class_weight='balanced'   # ← ADD THIS LINE
)

dt_model.fit(X_train, y_train)

print('Model trained with cost-sensitive learning!')
print(f'Tree depth: {dt_model.get_depth()}')
print(f'Number of leaves: {dt_model.get_n_leaves()}')

## 5. Make predictions and evaluate

In [ ]:
# Use the trained model to predict on the TEST set
y_pred = dt_model.predict(X_test)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f'=== DECISION TREE RESULTS ===')
print(f'Accuracy: {accuracy * 100:.2f}%')
print()
print('Detailed Report:')
print(classification_report(y_test, y_pred, target_names=class_names))

In [ ]:
from sklearn.metrics import matthews_corrcoef

# ── Multiclass Averaging Strategies ──────────────────────
print('=== MULTICLASS AVERAGING STRATEGIES ===')
print()

# Macro = treats all classes equally regardless of size
from sklearn.metrics import f1_score, precision_score, recall_score

macro_f1        = f1_score(y_test, y_pred, average='macro')
weighted_f1     = f1_score(y_test, y_pred, average='weighted')
macro_precision = precision_score(y_test, y_pred, average='macro')
macro_recall    = recall_score(y_test, y_pred, average='macro')

print(f'Macro F1     : {macro_f1:.4f}   ← treats all classes equally')
print(f'Weighted F1  : {weighted_f1:.4f}   ← weighted by class size')
print(f'Macro Precision: {macro_precision:.4f}')
print(f'Macro Recall   : {macro_recall:.4f}')

# MCC = best single metric for imbalanced multiclass
mcc = matthews_corrcoef(y_test, y_pred)
print(f'\nMCC Score    : {mcc:.4f}   ← best metric for imbalanced data')
print('  (MCC ranges from -1 to +1, closer to 1 is better)')

In [ ]:
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import label_binarize

# Binarize labels for multiclass ROC-AUC
# Converts [0,1,2] into [[1,0,0],[0,1,0],[0,0,1]]
y_test_bin = label_binarize(y_test, classes=[0, 1, 2])
y_prob = dt_model.predict_proba(X_test)

# ROC-AUC for multiclass (one-vs-rest)
roc_auc = roc_auc_score(y_test_bin, y_prob, 
                         multi_class='ovr', 
                         average='macro')

print(f'ROC-AUC Score (macro): {roc_auc:.4f}')
print('  (closer to 1.0 = better, 0.5 = random guessing)')

In [ ]:
from sklearn.metrics import precision_recall_curve, auc, roc_curve

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = ['#2196F3', '#4CAF50', '#FF5722']

for i, (cls_name, color) in enumerate(zip(class_names, colors)):
    
    # ── ROC Curve ──────────────────────────
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_prob[:, i])
    roc_auc_cls  = auc(fpr, tpr)
    axes[0].plot(fpr, tpr, color=color, 
                 label=f'{cls_name} (AUC={roc_auc_cls:.2f})')

    # ── PR Curve ───────────────────────────
    precision, recall, _ = precision_recall_curve(
                               y_test_bin[:, i], y_prob[:, i])
    pr_auc = auc(recall, precision)
    axes[1].plot(recall, precision, color=color,
                 label=f'{cls_name} (AUC={pr_auc:.2f})')

# ROC chart formatting
axes[0].plot([0,1],[0,1],'k--', label='Random')
axes[0].set_title('ROC Curve (One-vs-Rest)')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].legend()

# PR chart formatting
axes[1].set_title('Precision-Recall Curve')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].legend()

plt.suptitle('Decision Tree — ROC & PR Curves', fontsize=14)
plt.tight_layout()
plt.savefig('member1_roc_pr_curves.png')
plt.show()
print('ROC and PR curves saved!')

## 6. Confusion Matrix (visual)

In [ ]:
# Confusion matrix shows which classes are confused with each other
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names,
            yticklabels=class_names)
plt.title('Decision Tree — Confusion Matrix')
plt.ylabel('Actual Position')
plt.xlabel('Predicted Position')
plt.tight_layout()
plt.savefig('member1_confusion_matrix.png')  # Save for the report
plt.show()
print('Confusion matrix saved!')